# HW6 : 총액 regression model 을 고르고, API 로 내보낸다
이름 · 학번 : 24011300 박진우

## 단계 ① : 읽고 나누기
X = 수량 · 단가, y = 총액. `test_size=0.2`, `random_state=0`

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## 단계 ② : model 셋의 test R²
linear regression / linear regression + qp (= 수량 × 단가) / decision tree (제한 없음)

In [2]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])

lin  = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

print("linear regression        test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp   test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))
print("linear + qp coef :", lin2.coef_.round(3), " intercept :", round(lin2.intercept_, 1))

linear regression        test R² : 0.862
linear regression + qp   test R² : 1.0
decision tree (no limit) test R² : 0.994
linear + qp coef : [0. 0. 1.]  intercept : -0.0


## 단계 ③ 답
**linear regression 이 0.862 에 그친 이유 :** 총액 = 수량 × 단가 인 **곱셈** 관계인데, linear regression 은 `a × 수량 + b × 단가 + c` 처럼 각 열에 계수를 곱해 **더하는** 모양(평면)만 만들 수 있다. 그래서 곱셈의 곡면을 평면으로 흉내 낼 뿐이고 R² 가 0.862 에 머문다.

**qp 를 더하면 1.0 이 되는 이유 :** 수량 × 단가 를 미리 계산한 열 `qp` 를 feature 로 넣으면, 총액은 `qp` 에 대해 정확히 직선(총액 = 1 × qp + 0)이 된다. model 은 계수 [0, 0, 1], 절편 0 을 찾아 공식을 그대로 복원하므로 R² = 1.0 이다 (feature engineering).

## 단계 ④ : max_depth 별 training · test R²

In [3]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## 단계 ⑤ : cross-validation (cv=5)

In [4]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.3f}  std {s.std():.3f}")

max_depth 3     mean 0.845  std 0.019
max_depth 5     mean 0.961  std 0.003
max_depth 8     mean 0.992  std 0.000
max_depth 12    mean 0.995  std 0.000
max_depth None  mean 0.994  std 0.001


## 단계 ⑤ 결정과 이유
**max_depth = 8 을 고른다.** cross-validation 평균이 depth 8 에서 0.992, 12 에서 0.995, 제한 없음 0.994 로 8 부터는 거의 같고(차이 0.003), 표준편차도 0.000 이다. score 가 거의 같으면 단순한 쪽을 고른다 : depth 8 은 잎 250 개, depth 12 는 잎 781 개.

training R² 1.000 과 test R² 0.994 의 차이가 작은 이유 : 총액 = 수량 × 단가 라는 **규칙이 실제로 존재**하는 문제라서, training 을 거의 외워도 그 규칙이 test 에도 그대로 통하기 때문이다 (반품 예제처럼 test R² 가 꺾이지 않음).

## 단계 ⑥ : 고른 model 을 저장 (joblib.dump)
세 숫자(수량 40 · 단가 200,000)를 비교한 뒤, API 에는 linear regression + qp 를 올린다.

In [5]:
import joblib

order  = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
order2 = order.assign(qp=order["quantity"] * order["unit_price"])
tree8  = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)

print("linear + qp      →", int(round(lin2.predict(order2)[0])))
print("tree (depth 8)   →", int(round(tree8.predict(order)[0])))
print("quantity × price →", 40 * 200000)
joblib.dump(lin2, "sales_model.pkl")          # linear regression + qp

linear + qp      → 8000000
tree (depth 8)   → 7362301
quantity × price → 8000000


['sales_model.pkl']

## 단계 ⑦ 답
**API 에는 linear regression + qp 를 올린다.** 수량 40 · 단가 200,000 에 대해 linear + qp 는 **8,000,000**, 실제 곱도 **8,000,000** 으로 정확히 같다. 반면 decision tree (max_depth 8) 는 test R² 0.991 로 점수는 높지만 **7,362,301** 을 답해 약 64만 원(8%)이 틀린다. tree 는 training 에서 본 구간의 평균값(계단)으로만 답하므로, 데이터가 드문 범위나 training 범위 밖에서는 더 벗어난다 (CO2 time series 에서 tree 가 365 에서 평평해진 것과 같은 이유).

단, 서버에서도 학습 때처럼 `qp = quantity × unit_price` 열을 똑같이 만들어 넣어야 한다 (app.py 의 `x["qp"] = ...`). 사실 공식이 있는 문제이므로, 실제 서비스라면 model 없이 곱하는 편이 가장 단순하고 정확하다.